# ira-esm-tokenizer, the ablation run

Trains three tokenizers that differ in exactly one thing each, then scores all
three side by side. Everything is set up to run **unattended**.

## How to run this

**Save Version -> Save & Run All (Commit).** Then close the laptop.

That runs the whole notebook server-side as a batch job and saves the output
when it finishes. An interactive session does not survive a dropped connection
or an idle laptop, and two checkpoints have already been lost that way.

The training cells below run in the **foreground** for exactly this reason. In
a committed run a backgrounded `nohup` process gets killed the moment the
notebook reaches its last cell, so the notebook has to wait for training rather
than launching it and moving on.

## Before you commit

Attach both datasets in the sidebar.

- **ira-esm-tokenizer-full**, the `.py` source, 15 files
- **ira-esm-parsed**, the 921 parsed structures

Set the accelerator to **GPU T4 x2**. Two runs go in parallel, one per card.

Expect roughly two and a quarter hours. Two runs at about 65 minutes, then a
third.

In [ ]:
import torch, subprocess

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
GPUS = torch.cuda.device_count()
print("GPUs visible:", GPUS)
if GPUS:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())

# Not fatal, but a committed run on CPU would take days rather than hours, so
# it is better to fail here than to burn a batch job discovering it.
assert GPUS >= 1, "no GPU. Set Accelerator to GPU T4 x2 before committing."

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
ROOT  = Path("/kaggle/working/ira-esm-tokenizer")

EXPECTED = {"train.py", "analyze_codebook.py", "compare_esm3.py", "compare_runs.py",
            "geometry.py", "encoder.py", "quantizer.py", "decoder.py",
            "dataset.py", "download_pdbs.py", "fetch_pdb_ids.py", "parse_structures.py"}

zip_path = next(INPUT.rglob("*.zip"), None)
if zip_path is not None:
    zipfile.ZipFile(zip_path).extractall(ROOT.parent)
    print("unzipped", zip_path)
else:
    # More than one attached dataset contains a folder that LOOKS like the
    # project, because ira-esm-parsed was built from an old notebook output and
    # carries a stale copy of data/*.py. So rather than taking the first match,
    # score every candidate by how many expected files it actually has and take
    # the best one. Picking the stale copy would train the wrong code.
    candidates = {}
    for marker in INPUT.rglob("*.py"):
        root = marker.parent
        if root.name in ("data", "model"):
            root = root.parent
        candidates.setdefault(root, set()).add(marker.name)

    scored = sorted(candidates.items(), key=lambda kv: -len(kv[1] & EXPECTED))
    if not scored or not (scored[0][1] & EXPECTED):
        raise SystemExit(f"no project source under {INPUT}. Attach ira-esm-tokenizer-full.")

    best, files = scored[0]
    print("source candidates, best first:")
    for root, found in scored[:4]:
        print(f"  {len(found & EXPECTED):2d}/{len(EXPECTED)}  {root}")
    shutil.copytree(best, ROOT, dirs_exist_ok=True)
    print("\nusing", best)

os.chdir(ROOT); sys.path.insert(0, str(ROOT))

found = {p.name for p in ROOT.rglob("*.py")}
missing = EXPECTED - found
assert not missing, f"source is incomplete, missing: {sorted(missing)}"
print("ready:", len(found), "files")

In [ ]:
from pathlib import Path
import shutil

target = Path("data/parsed"); target.mkdir(parents=True, exist_ok=True)

# Search all of /kaggle/input rather than a fixed path. Kaggle mounts datasets
# at /kaggle/input/<slug> in some accounts and /kaggle/input/datasets/<user>/
# <slug> in others, and a dataset built from a notebook output keeps that
# output's folder nesting on top. Nothing else here produces .npz files.
found = list(Path("/kaggle/input").rglob("*.npz"))
assert found, "no .npz under /kaggle/input. Attach ira-esm-parsed."
print("found", len(found), "in", found[0].parent)

for f in found:
    if not (target / f.name).exists():
        shutil.copy(f, target / f.name)

n = len(list(target.glob("*.npz")))
print(f"{n} structures ready")
assert n > 800, f"expected ~921, got {n}"

In [ ]:
import torch
from model.encoder import StructureEncoder
from model.quantizer import VectorQuantizer
from model.decoder import StructureDecoder, fape_loss

# collate_fn pads with zero coordinates. Without the clamps in build_frames
# that produces NaN which spreads into the REAL residues too. Two hours of GPU
# time is worth ten seconds of checking that the loaded code is the fixed one.
torch.manual_seed(0)
B, L = 2, 12
coords = torch.randn(B, L, 4, 3) * 5
mask = torch.ones(B, L, dtype=torch.bool)
coords[1, 7:] = 0.0
mask[1, 7:] = False

for nb in (0, 16):
    enc, vq, dec = StructureEncoder(neighbours=nb), VectorQuantizer(), StructureDecoder()
    z = enc(coords, mask); q = vq(z, mask)
    loss = fape_loss(dec(q["quantized"], mask), coords, mask) + q["loss"]
    assert torch.isfinite(loss), f"NaN loss at neighbours={nb}"
    assert not torch.isnan(z[mask]).any(), f"NaN in real residues at neighbours={nb}"
    print(f"neighbours={nb:2d}  padded-batch loss {loss.item():.4f}  finite")

## Train the three runs

One variable each, everything else identical.

| Run | Change | Question it answers |
|---|---|---|
| `4096` | none, this is the baseline | what the current design does |
| `512` | `--num-codes 512` | does scarcity force codes to be reused |
| `knn16` | `--neighbours 16` | does ESM-3's receptive field make tokens transferable |

`--seed` and `--val-fraction` stay at their defaults everywhere, so all three
are validated on the same 92 structures and the comparison is meaningful.

Runs go two at a time, one per GPU. The cell blocks until every run finishes,
which is what makes this work as a committed job.

In [ ]:
import os, subprocess, time
from pathlib import Path

ROOT = Path("/kaggle/working/ira-esm-tokenizer")
LOGS = Path("/kaggle/working"); COMMON = [
    "--parsed-dir", "data/parsed",
    "--epochs", "200", "--max-length", "256", "--budget", "262144",
    "--lr", "3e-4", "--num-workers", "2", "--revive-every", "200",
]

RUNS = [
    ("4096",  []),                       # baseline
    ("512",   ["--num-codes", "512"]),
    ("knn16", ["--neighbours", "16"]),
]

def launch(name, extra, gpu):
    ckpt = Path(f"/kaggle/working/checkpoints-{name}")
    ckpt.mkdir(parents=True, exist_ok=True)
    log = open(LOGS / f"train-{name}.log", "w")
    print(f"  starting {name} on GPU {gpu} -> {ckpt}", flush=True)
    return name, log, subprocess.Popen(
        ["python", "-u", "train.py", "--checkpoint-dir", str(ckpt), *COMMON, *extra],
        cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
        env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
    )

# In waves of one-per-GPU. Two runs sharing a card would fit in 16GB but would
# also slow each other down and risk an out-of-memory late in a two-hour job,
# which is a bad trade for finishing a little sooner.
import torch
n_gpus = max(1, torch.cuda.device_count())
started = time.time()

for wave in [RUNS[i:i + n_gpus] for i in range(0, len(RUNS), n_gpus)]:
    print(f"\nwave: {[n for n, _ in wave]}", flush=True)
    procs = [launch(name, extra, gpu) for gpu, (name, extra) in enumerate(wave)]
    for name, log, p in procs:
        code = p.wait()
        log.close()
        # Report rather than raise. If one run fails we still want the others,
        # and a two-hour job should not be thrown away over one bad config.
        print(f"  {name} finished, exit {code}, {(time.time()-started)/60:.0f} min elapsed", flush=True)

print("\nall runs done")
for name, _ in RUNS:
    print(f"\n--- {name} ---")
    print(subprocess.run(["tail", "-n", "3", f"/kaggle/working/train-{name}.log"],
                         capture_output=True, text=True).stdout)

## Score every run

Per-run reports first, then the table that puts them side by side.

Read the ablation table on **two axes**, not one. Reconstruction error alone
will pick the wrong tokenizer every time, because giving each residue a
near-unique code makes the decoder's job easier. That is exactly how the
4096-code run ended up at 4.1 residues per code.

A run that improves `transplant agreement`, `codes per local shape` and
`NMI(code, protein)` while barely moving `val reconstruction` is a real gain.
One that improves them because reconstruction collapsed is not.

In [ ]:
import subprocess, sys
from pathlib import Path

# Per-run reports first, then the table that puts them side by side.
for name in ["4096", "512", "knn16"]:
    ckpt = Path(f"/kaggle/working/checkpoints-{name}/best.pt")
    if not ckpt.exists():
        print("skipping", name, "(no checkpoint)")
        continue
    print("\n" + "#" * 70); print("#", name); print("#" * 70, flush=True)
    for script, extra in [("analyze_codebook.py", ["--max-structures", "200", "--plots"]),
                          ("compare_esm3.py", ["--max-structures", "100"])]:
        subprocess.run([sys.executable, script,
                        "--checkpoint", str(ckpt),
                        "--parsed-dir", "data/parsed",
                        "--out-dir", f"/kaggle/working/analysis/{name}", *extra], check=False)

In [ ]:
import subprocess, sys
from pathlib import Path

runs = []
for name in ["4096", "512", "knn16"]:
    p = Path(f"/kaggle/working/checkpoints-{name}/best.pt")
    if p.exists():
        runs += ["--run", f"{name}={p}"]
    else:
        print("not available, skipping:", name)

assert len(runs) >= 4, "need at least two finished runs"

subprocess.run([sys.executable, "compare_runs.py", *runs,
                "--parsed-dir", "data/parsed",
                "--max-structures", "100",
                "--out", "/kaggle/working/analysis/ablation.json"], check=False)

In [ ]:
import re
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
PATTERN = (r"epoch\s+(\d+)\s+train ([\d.]+)A\s+val ([\d.]+)A\s+vq ([\d.]+)\s+"
           r"codes (\d+)/\d+\s+ppl (\d+)")

for name in ["4096", "512", "knn16"]:
    try:
        rows = [[float(x) for x in m.groups()]
                for line in open(f"/kaggle/working/train-{name}.log")
                if (m := re.match(PATTERN, line))]
    except FileNotFoundError:
        continue
    if not rows:
        continue
    epoch, train, val, vq, codes, ppl = zip(*rows)
    axes[0].plot(epoch, val, label=name)
    axes[1].plot(epoch, vq, label=name)
    axes[2].plot(epoch, ppl, label=name)
    print(f"{name:>6}  best val {min(val):.3f}A at epoch {int(epoch[val.index(min(val))])}")

axes[0].set_title("validation reconstruction"); axes[0].set_ylabel("Angstroms")
axes[1].set_title("VQ loss"); axes[1].set_yscale("log")
# Perplexity is on very different scales across runs (512 vs 4096 codes), so a
# log axis is the only way to see all three curves at once.
axes[2].set_title("codebook perplexity"); axes[2].set_yscale("log")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()